In [1]:
"""
Attention-Based Fusion Baseline

Compares multi-agent fusion vs single attention-based model
Tests hypothesis: Explicit multi-agent > Implicit attention
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from pathlib import Path
import json
from tqdm.auto import tqdm

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("   Running on CPU (this will be slower but works fine)")

# Disease list
DISEASE_LIST = [
    'SEPSIS', 'PNEUMONIA', 'RESPIRATORY_FAILURE',
    'ACUTE_KIDNEY_INJURY', 'HEART_FAILURE', 'ATRIAL_FIBRILLATION',
    'CORONARY_ARTERY_DISEASE', 'ANEMIA', 'PANCREATITIS'
]

print(f"\n✅ Setup complete - {len(DISEASE_LIST)} diseases")

🖥️ Using device: cpu
   Running on CPU (this will be slower but works fine)

✅ Setup complete - 9 diseases


In [2]:
print("="*70)
print("📂 LOADING DATA")
print("="*70)

# Load fusion data (33 features)
X_fusion = np.load('../../data/processed/X_fusion_val.npy')
print(f"✅ X_fusion: {X_fusion.shape}")

# Extract modalities
agent1_pred = X_fusion[:, 0:9]    # Labs (9 probs)
agent2_pred = X_fusion[:, 9:18]   # Notes (9 probs)
agent3_pred = X_fusion[:, 18:27]  # Vitals (9 probs)
meta_features = X_fusion[:, 27:33]  # Confidence + availability

print(f"\nModalities:")
print(f"  Agent 1 (Labs):   {agent1_pred.shape}")
print(f"  Agent 2 (Notes):  {agent2_pred.shape}")
print(f"  Agent 3 (Vitals): {agent3_pred.shape}")
print(f"  Meta features:    {meta_features.shape}")

# Load labels
y_labels = {}
for disease in DISEASE_LIST:
    disease_filename = disease.lower()
    y_labels[disease] = np.load(f'../../data/processed/y_fusion_val_{disease_filename}.npy')

# Stack all labels
y_all = np.column_stack([y_labels[d] for d in DISEASE_LIST])
print(f"\n✅ Labels: {y_all.shape} (patients, diseases)")

# Create SAME split as fusion
indices = np.arange(len(X_fusion))
train_val_idx, test_idx = train_test_split(indices, test_size=0.30, random_state=42)
train_idx, val_idx = train_test_split(train_val_idx, test_size=0.20/(0.50+0.20), random_state=42)

print(f"\n📊 Splits:")
print(f"  Train: {len(train_idx)} ({len(train_idx)/len(X_fusion)*100:.1f}%)")
print(f"  Val:   {len(val_idx)} ({len(val_idx)/len(X_fusion)*100:.1f}%)")
print(f"  Test:  {len(test_idx)} ({len(test_idx)/len(X_fusion)*100:.1f}%)")

📂 LOADING DATA
✅ X_fusion: (4643, 33)

Modalities:
  Agent 1 (Labs):   (4643, 9)
  Agent 2 (Notes):  (4643, 9)
  Agent 3 (Vitals): (4643, 9)
  Meta features:    (4643, 6)

✅ Labels: (4643, 9) (patients, diseases)

📊 Splits:
  Train: 2321 (50.0%)
  Val:   929 (20.0%)
  Test:  1393 (30.0%)


In [3]:
class MultiModalDataset(Dataset):
    """Dataset for attention-based model"""
    
    def __init__(self, agent1, agent2, agent3, labels):
        self.agent1 = torch.FloatTensor(agent1)
        self.agent2 = torch.FloatTensor(agent2)
        self.agent3 = torch.FloatTensor(agent3)
        self.labels = torch.FloatTensor(labels)
        
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return {
            'agent1': self.agent1[idx],
            'agent2': self.agent2[idx],
            'agent3': self.agent3[idx],
            'labels': self.labels[idx]
        }

# Create datasets
train_dataset = MultiModalDataset(
    agent1_pred[train_idx],
    agent2_pred[train_idx],
    agent3_pred[train_idx],
    y_all[train_idx]
)

val_dataset = MultiModalDataset(
    agent1_pred[val_idx],
    agent2_pred[val_idx],
    agent3_pred[val_idx],
    y_all[val_idx]
)

test_dataset = MultiModalDataset(
    agent1_pred[test_idx],
    agent2_pred[test_idx],
    agent3_pred[test_idx],
    y_all[test_idx]
)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

print(f"✅ Datasets created:")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches:   {len(val_loader)}")
print(f"   Test batches:  {len(test_loader)}")

✅ Datasets created:
   Train batches: 37
   Val batches:   8
   Test batches:  11


In [4]:
class CrossModalAttentionFusion(nn.Module):
    """
    Attention-based fusion model
    
    Architecture:
    1. Each modality gets projected to common space
    2. Cross-modal attention learns weights
    3. Weighted fusion
    4. Output layer
    """
    
    def __init__(self, input_dim=9, hidden_dim=64, n_diseases=9):
        super().__init__()
        
        # Modality encoders (project to common space)
        self.encoder1 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.encoder2 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.encoder3 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        # Attention mechanism
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=4,
            dropout=0.1,
            batch_first=True
        )
        
        # Fusion layers
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, n_diseases)
        )
        
    def forward(self, agent1, agent2, agent3):
        batch_size = agent1.size(0)
        
        # Encode each modality
        h1 = self.encoder1(agent1)  # (batch, hidden)
        h2 = self.encoder2(agent2)
        h3 = self.encoder3(agent3)
        
        # Stack modalities: (batch, 3 modalities, hidden)
        modalities = torch.stack([h1, h2, h3], dim=1)
        
        # Self-attention across modalities
        # Query, Key, Value are all the modalities
        attended, attention_weights = self.attention(
            modalities, modalities, modalities
        )
        
        # Global average pooling across modalities
        fused = attended.mean(dim=1)  # (batch, hidden)
        
        # Output
        output = self.fusion(fused)  # (batch, n_diseases)
        
        return torch.sigmoid(output), attention_weights

# Create model
model = CrossModalAttentionFusion(
    input_dim=9,
    hidden_dim=64,
    n_diseases=9
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("="*70)
print("🤖 ATTENTION-BASED FUSION MODEL")
print("="*70)
print(f"\nArchitecture:")
print(f"  Input:  9 probabilities per modality (3 modalities)")
print(f"  Hidden: 64 dimensions")
print(f"  Attention: 4 heads")
print(f"  Output: 9 disease probabilities")
print(f"\nParameters:")
print(f"  Total:      {total_params:,}")
print(f"  Trainable:  {trainable_params:,}")
print(f"\nDevice: {device}")

print("\n" + str(model))

🤖 ATTENTION-BASED FUSION MODEL

Architecture:
  Input:  9 probabilities per modality (3 modalities)
  Hidden: 64 dimensions
  Attention: 4 heads
  Output: 9 disease probabilities

Parameters:
  Total:      23,305
  Trainable:  23,305

Device: cpu

CrossModalAttentionFusion(
  (encoder1): Sequential(
    (0): Linear(in_features=9, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
  )
  (encoder2): Sequential(
    (0): Linear(in_features=9, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
  )
  (encoder3): Sequential(
    (0): Linear(in_features=9, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
  )
  (attention): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
  )
  (fusion): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_feat

In [5]:
def train_epoch(model, loader, optimizer, criterion, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    
    for batch in tqdm(loader, desc="Training", leave=False):
        agent1 = batch['agent1'].to(device)
        agent2 = batch['agent2'].to(device)
        agent3 = batch['agent3'].to(device)
        labels = batch['labels'].to(device)
        
        # Forward
        predictions, _ = model(agent1, agent2, agent3)
        loss = criterion(predictions, labels)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

def evaluate(model, loader, device):
    """Evaluate model"""
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating", leave=False):
            agent1 = batch['agent1'].to(device)
            agent2 = batch['agent2'].to(device)
            agent3 = batch['agent3'].to(device)
            labels = batch['labels'].to(device)
            
            predictions, _ = model(agent1, agent2, agent3)
            
            all_preds.append(predictions.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    
    preds = np.vstack(all_preds)
    labels = np.vstack(all_labels)
    
    # Calculate metrics per disease
    aucs = []
    for i in range(9):
        auc = roc_auc_score(labels[:, i], preds[:, i])
        aucs.append(auc)
    
    avg_auc = np.mean(aucs)
    
    return avg_auc, aucs, preds, labels

print("✅ Training functions defined")

✅ Training functions defined


In [6]:
print("="*70)
print("🔨 TRAINING ATTENTION-BASED MODEL")
print("="*70)

# Loss and optimizer
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3, verbose=True
)

# Training loop
n_epochs = 30
best_val_auc = 0
patience = 5
patience_counter = 0

history = {
    'train_loss': [],
    'val_auc': []
}

print(f"\nTraining for up to {n_epochs} epochs...")
print(f"Early stopping patience: {patience}")
print()

for epoch in range(n_epochs):
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    
    # Validate
    val_auc, val_aucs, _, _ = evaluate(model, val_loader, device)
    
    # Update history
    history['train_loss'].append(train_loss)
    history['val_auc'].append(val_auc)
    
    # Learning rate schedule
    scheduler.step(val_auc)
    
    # Print progress
    print(f"Epoch {epoch+1:2d}/{n_epochs} | "
          f"Loss: {train_loss:.4f} | "
          f"Val AUC: {val_auc:.3f}")
    
    # Save best model
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        Path('../../models/attention').mkdir(parents=True, exist_ok=True)
        torch.save(model.state_dict(), '../../models/attention/attention_fusion_best.pt')
        patience_counter = 0
        print(f"  ✅ New best! Saved model")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\n⏹️ Early stopping after {epoch+1} epochs")
            break

print("\n" + "="*70)
print("✅ TRAINING COMPLETE")
print("="*70)
print(f"Best validation AUC: {best_val_auc:.3f}")

🔨 TRAINING ATTENTION-BASED MODEL

Training for up to 30 epochs...
Early stopping patience: 5



C:\Users\athar\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Training:   0%|          | 0/37 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

Epoch  1/30 | Loss: 0.6381 | Val AUC: 0.430
  ✅ New best! Saved model


...

Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

Epoch 29/30 | Loss: 0.3839 | Val AUC: 0.872


Training:   0%|          | 0/37 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

Epoch 30/30 | Loss: 0.3832 | Val AUC: 0.873
  ✅ New best! Saved model

✅ TRAINING COMPLETE
Best validation AUC: 0.873


In [7]:
print("\n" + "="*70)
print("🧪 EVALUATING ON TEST SET")
print("="*70)

# Load best model
model.load_state_dict(torch.load('../../models/attention/attention_fusion_best.pt'))
print("✅ Loaded best model")

# Evaluate
test_auc, test_aucs, test_preds, test_labels = evaluate(model, test_loader, device)

print(f"\n📊 Test Set Performance:")
print(f"{'Disease':30s} {'AUC':>10s}")
print("-" * 42)

for i, disease in enumerate(DISEASE_LIST):
    print(f"{disease:30s} {test_aucs[i]:>10.3f}")

print("-" * 42)
print(f"{'AVERAGE':30s} {test_auc:>10.3f}")

# Save results
attention_results = pd.DataFrame({
    'Disease': DISEASE_LIST,
    'AUC': test_aucs
})
attention_results.to_csv('../../results/attention_baseline_results.csv', index=False)

print(f"\n✅ Saved results to: ../results/attention_baseline_results.csv")


🧪 EVALUATING ON TEST SET
✅ Loaded best model


Evaluating:   0%|          | 0/11 [00:00<?, ?it/s]


📊 Test Set Performance:
Disease                               AUC
------------------------------------------
SEPSIS                              0.865
PNEUMONIA                           0.876
RESPIRATORY_FAILURE                 0.885
ACUTE_KIDNEY_INJURY                 0.936
HEART_FAILURE                       0.900
ATRIAL_FIBRILLATION                 0.871
CORONARY_ARTERY_DISEASE             0.920
ANEMIA                              0.830
PANCREATITIS                        0.792
------------------------------------------
AVERAGE                             0.875

✅ Saved results to: ../results/attention_baseline_results.csv


In [13]:
# ============================================================================
# CELL 7B: Calculate F1 Scores for Attention Model
# ============================================================================
print("\n" + "="*70)
print("📊 CALCULATING F1 SCORES FOR ATTENTION MODEL")
print("="*70)

# We have test_preds and test_labels from Cell 7
# Need to convert probabilities to binary predictions

# Load optimal thresholds from fusion (for fair comparison)
with open('../../results/optimal_thresholds.json', 'r') as f:
    optimal_thresholds = json.load(f)

print("\nUsing same thresholds as fusion for fair comparison:\n")

attention_f1_results = []

for i, disease in enumerate(DISEASE_LIST):
    y_true = test_labels[:, i]
    y_pred_proba = test_preds[:, i]
    
    # Get threshold
    threshold = optimal_thresholds.get(disease, 0.5)
    y_pred_binary = (y_pred_proba >= threshold).astype(int)
    
    # Calculate metrics
    auc = test_aucs[i]
    f1 = f1_score(y_true, y_pred_binary)
    precision = precision_score(y_true, y_pred_binary, zero_division=0)
    recall = recall_score(y_true, y_pred_binary, zero_division=0)
    
    attention_f1_results.append({
        'Disease': disease,
        'AUC': auc,
        'F1': f1,
        'Precision': precision,
        'Recall': recall,
        'Threshold': threshold
    })
    
    print(f"{disease:30s} AUC: {auc:.3f}, F1: {f1:.3f}, Prec: {precision:.3f}, Rec: {recall:.3f}")

attention_f1_df = pd.DataFrame(attention_f1_results)

print(f"\n📊 Attention Model Averages:")
print(f"   AUC:       {attention_f1_df['AUC'].mean():.3f}")
print(f"   F1:        {attention_f1_df['F1'].mean():.3f}")
print(f"   Precision: {attention_f1_df['Precision'].mean():.3f}")
print(f"   Recall:    {attention_f1_df['Recall'].mean():.3f}")

# Save
attention_f1_df.to_csv('../../results/attention_f1_results.csv', index=False)
print(f"\n✅ Saved F1 results: ../results/attention_f1_results.csv")


📊 CALCULATING F1 SCORES FOR ATTENTION MODEL

Using same thresholds as fusion for fair comparison:

SEPSIS                         AUC: 0.865, F1: 0.523, Prec: 0.686, Rec: 0.423
PNEUMONIA                      AUC: 0.876, F1: 0.657, Prec: 0.590, Rec: 0.740
RESPIRATORY_FAILURE            AUC: 0.885, F1: 0.760, Prec: 0.788, Rec: 0.734
ACUTE_KIDNEY_INJURY            AUC: 0.936, F1: 0.834, Prec: 0.813, Rec: 0.857
HEART_FAILURE                  AUC: 0.900, F1: 0.766, Prec: 0.703, Rec: 0.842
ATRIAL_FIBRILLATION            AUC: 0.871, F1: 0.714, Prec: 0.689, Rec: 0.742
CORONARY_ARTERY_DISEASE        AUC: 0.920, F1: 0.790, Prec: 0.797, Rec: 0.783
ANEMIA                         AUC: 0.830, F1: 0.664, Prec: 0.650, Rec: 0.678
PANCREATITIS                   AUC: 0.792, F1: 0.418, Prec: 0.329, Rec: 0.571

📊 Attention Model Averages:
   AUC:       0.875
   F1:        0.681
   Precision: 0.672
   Recall:    0.708

✅ Saved F1 results: ../results/attention_f1_results.csv


In [14]:
print("\n" + "="*70)
print("⚔️ COMPARISON: ATTENTION vs MULTI-AGENT FUSION")
print("="*70)

# Load multi-agent fusion results
fusion_results = pd.read_csv('../../results/fusion_test_results_optimized.csv')

# Align disease names
fusion_aucs = []
for disease in DISEASE_LIST:
    row = fusion_results[fusion_results['Unnamed: 0'] == disease]
    if len(row) > 0:
        fusion_aucs.append(row['AUC'].values[0])
    else:
        fusion_aucs.append(np.nan)

# Create comparison
comparison = pd.DataFrame({
    'Disease': DISEASE_LIST,
    'Attention_AUC': test_aucs,
    'Fusion_AUC': fusion_aucs,
    'Difference': np.array(fusion_aucs) - np.array(test_aucs)
})

print(f"\n{'Disease':30s} {'Attention':>12s} {'Multi-Agent':>12s} {'Winner':>12s} {'Diff':>10s}")
print("=" * 80)

fusion_wins = 0
attention_wins = 0

for _, row in comparison.iterrows():
    disease = row['Disease']
    att_auc = row['Attention_AUC']
    fus_auc = row['Fusion_AUC']
    diff = row['Difference']
    
    if fus_auc > att_auc:
        winner = "Fusion ✅"
        fusion_wins += 1
    elif att_auc > fus_auc:
        winner = "Attention"
        attention_wins += 1
    else:
        winner = "Tie"
    
    print(f"{disease:30s} {att_auc:12.3f} {fus_auc:12.3f} {winner:>12s} {diff:+10.3f}")

print("=" * 80)
avg_attention = comparison['Attention_AUC'].mean()
avg_fusion = comparison['Fusion_AUC'].mean()
avg_diff = avg_fusion - avg_attention

print(f"{'AVERAGE':30s} {avg_attention:12.3f} {avg_fusion:12.3f} {'':>12s} {avg_diff:+10.3f}")

print(f"\n📊 Win Record:")
print(f"  Multi-Agent Fusion: {fusion_wins}/9 ({fusion_wins/9*100:.1f}%)")
print(f"  Attention Baseline: {attention_wins}/9 ({attention_wins/9*100:.1f}%)")

# Statistical test
from scipy import stats
t_stat, p_value = stats.ttest_rel(fusion_aucs, test_aucs)

print(f"\n📈 Statistical Significance:")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value:     {p_value:.6f}")
print(f"  Significant: {'YES ✅' if p_value < 0.05 else 'NO ❌'} (α=0.05)")

# Save comparison
comparison.to_csv('../../results/attention_vs_fusion_comparison.csv', index=False)
print(f"\n✅ Saved comparison: ../results/attention_vs_fusion_comparison.csv")

# ============================================================================
# F1 SCORE COMPARISON
# ============================================================================
print("\n" + "="*70)
print("📊 F1 SCORE COMPARISON (The Clinical Impact Metric)")
print("="*70)

# Load fusion F1 scores
fusion_f1_results = pd.read_csv('../../results/fusion_test_results_optimized.csv')

fusion_f1_scores = []
for disease in DISEASE_LIST:
    row = fusion_f1_results[fusion_f1_results['Unnamed: 0'] == disease]
    if len(row) > 0:
        fusion_f1_scores.append(row['F1_Optimized'].values[0])
    else:
        fusion_f1_scores.append(np.nan)

attention_f1_scores = attention_f1_df['F1'].values

# Create F1 comparison
f1_comparison = pd.DataFrame({
    'Disease': DISEASE_LIST,
    'Attention_F1': attention_f1_scores,
    'Fusion_F1': fusion_f1_scores,
    'F1_Difference': np.array(fusion_f1_scores) - np.array(attention_f1_scores)
})

print(f"\n{'Disease':30s} {'Attention':>12s} {'Multi-Agent':>12s} {'Winner':>12s} {'Diff':>10s}")
print("=" * 80)

fusion_f1_wins = 0
attention_f1_wins = 0

for _, row in f1_comparison.iterrows():
    disease = row['Disease']
    att_f1 = row['Attention_F1']
    fus_f1 = row['Fusion_F1']
    diff = row['F1_Difference']
    
    if fus_f1 > att_f1:
        winner = "Fusion ✅"
        fusion_f1_wins += 1
    elif att_f1 > fus_f1:
        winner = "Attention"
        attention_f1_wins += 1
    else:
        winner = "Tie"
    
    print(f"{disease:30s} {att_f1:12.3f} {fus_f1:12.3f} {winner:>12s} {diff:+10.3f}")

print("=" * 80)
avg_attention_f1 = f1_comparison['Attention_F1'].mean()
avg_fusion_f1 = f1_comparison['Fusion_F1'].mean()
avg_f1_diff = avg_fusion_f1 - avg_attention_f1

print(f"{'AVERAGE':30s} {avg_attention_f1:12.3f} {avg_fusion_f1:12.3f} {'':>12s} {avg_f1_diff:+10.3f}")

print(f"\n📊 F1 Win Record:")
print(f"  Multi-Agent Fusion: {fusion_f1_wins}/9 ({fusion_f1_wins/9*100:.1f}%)")
print(f"  Attention Baseline: {attention_f1_wins}/9 ({attention_f1_wins/9*100:.1f}%)")

# Statistical test on F1
t_stat_f1, p_value_f1 = stats.ttest_rel(fusion_f1_scores, attention_f1_scores)

print(f"\n📈 F1 Statistical Significance:")
print(f"  t-statistic: {t_stat_f1:.4f}")
print(f"  p-value:     {p_value_f1:.6f}")
print(f"  Significant: {'YES ✅' if p_value_f1 < 0.05 else 'NO ❌'} (α=0.05)")

print(f"\n💡 INTERPRETATION:")
print(f"   AUC improvement:  +{avg_diff:.3f} ({avg_diff/avg_attention*100:+.1f}%)")
print(f"   F1 improvement:   +{avg_f1_diff:.3f} ({avg_f1_diff/avg_attention_f1*100:+.1f}%)")
print(f"\n   → F1 gap is {'LARGER' if abs(avg_f1_diff) > abs(avg_diff) else 'smaller'} than AUC gap")
print(f"   → Attention baseline struggles even more at actual classification!")

# Save F1 comparison
f1_comparison.to_csv('../../results/attention_vs_fusion_f1_comparison.csv', index=False)
print(f"\n✅ Saved F1 comparison: ../results/attention_vs_fusion_f1_comparison.csv")


⚔️ COMPARISON: ATTENTION vs MULTI-AGENT FUSION

Disease                           Attention  Multi-Agent       Winner       Diff
SEPSIS                                0.865        0.906     Fusion ✅     +0.042
PNEUMONIA                             0.876        0.885     Fusion ✅     +0.009
RESPIRATORY_FAILURE                   0.885        0.897     Fusion ✅     +0.012
ACUTE_KIDNEY_INJURY                   0.936        0.938     Fusion ✅     +0.002
HEART_FAILURE                         0.900        0.906     Fusion ✅     +0.006
ATRIAL_FIBRILLATION                   0.871        0.887     Fusion ✅     +0.016
CORONARY_ARTERY_DISEASE               0.920        0.924     Fusion ✅     +0.004
ANEMIA                                0.830        0.850     Fusion ✅     +0.020
PANCREATITIS                          0.792        0.835     Fusion ✅     +0.043
AVERAGE                               0.875        0.892                  +0.017

📊 Win Record:
  Multi-Agent Fusion: 9/9 (100.0%)
  Attentio

In [11]:
print("\n" + "="*70)
print("🔍 ANALYZING LEARNED ATTENTION WEIGHTS")
print("="*70)

# Get attention weights for test set
model.eval()
all_attention_weights = []

with torch.no_grad():
    for batch in test_loader:
        agent1 = batch['agent1'].to(device)
        agent2 = batch['agent2'].to(device)
        agent3 = batch['agent3'].to(device)
        
        _, attention_weights = model(agent1, agent2, agent3)
        # attention_weights shape: (batch, num_heads, 3, 3)
        all_attention_weights.append(attention_weights.cpu().numpy())

# Concatenate along batch dimension (axis 0)
attention_all = np.concatenate(all_attention_weights, axis=0)

print(f"Collected attention weights: {attention_all.shape}")

# Average across all patients only (keep the 3x3 matrix)
attention_matrix = attention_all.mean(axis=0)  # Shape: (3, 3)

print(f"Attention matrix shape: {attention_matrix.shape}")

print("\n📊 Average Attention Matrix (how each modality attends to others):")
print(f"\n{'':12s} {'Labs':>12s} {'Notes':>12s} {'Vitals':>12s}")
print("-" * 50)
modalities = ['Labs', 'Notes', 'Vitals']
for i, mod in enumerate(modalities):
    print(f"{mod:12s}", end='')
    for j in range(3):
        print(f"{attention_matrix[i,j]:12.3f}", end='')
    print()

print("\n💡 Interpretation:")
print("  Diagonal values: Self-attention (how much each modality focuses on itself)")
print("  Off-diagonal: Cross-attention (how much one modality depends on others)")
print()
print("  High values (>0.4): Strong attention/dependency")
print("  Low values (<0.3): Weak attention")

# Visualize which modality the model relies on most
modality_importance = attention_matrix.sum(axis=0)  # Sum across queries (rows)
print(f"\n📊 Overall Modality Importance (based on attention received):")
for i, mod in enumerate(modalities):
    print(f"  {mod:10s} {modality_importance[i]:.3f}")
    
most_important = modalities[np.argmax(modality_importance)]
print(f"\n  → Most attended modality: {most_important}")

print("\n🔍 Cross-Modal Dependencies:")
for i, mod_query in enumerate(modalities):
    # Find which other modality this one attends to most
    cross_attention = [attention_matrix[i,j] for j in range(3) if j != i]
    other_mods = [modalities[j] for j in range(3) if j != i]
    max_idx = np.argmax(cross_attention)
    print(f"  {mod_query:10s} relies most on: {other_mods[max_idx]:10s} (attention: {cross_attention[max_idx]:.3f})")


🔍 ANALYZING LEARNED ATTENTION WEIGHTS
Collected attention weights: (1393, 3, 3)
Attention matrix shape: (3, 3)

📊 Average Attention Matrix (how each modality attends to others):

                     Labs        Notes       Vitals
--------------------------------------------------
Labs               0.326       0.279       0.395
Notes              0.333       0.277       0.390
Vitals             0.343       0.289       0.368

💡 Interpretation:
  Diagonal values: Self-attention (how much each modality focuses on itself)
  Off-diagonal: Cross-attention (how much one modality depends on others)

  High values (>0.4): Strong attention/dependency
  Low values (<0.3): Weak attention

📊 Overall Modality Importance (based on attention received):
  Labs       1.002
  Notes      0.844
  Vitals     1.153

  → Most attended modality: Vitals

🔍 Cross-Modal Dependencies:
  Labs       relies most on: Vitals     (attention: 0.395)
  Notes      relies most on: Vitals     (attention: 0.390)
  Vitals   